# <font color="brown">Product Recommendation Case Study (SVD) </font>

## <font color = "brown">Problem Statement </font>

### <font color="blue"> Context

The bank has 300 customers and 41 products (the same product catalog from our Hierarchical Clustering case study), but any one customer has only ever tried a handful of them, on average, just over a third of possible customer-product pairs have any data at all. The cross-sell team wants to recommend products customers *haven't* tried yet, but are likely to want, based on patterns in what similar customers have rated highly.

### <font color="blue"> Objective

- Can we predict how a customer would rate a product they've never held, purely from the ratings we *do* have, from them and from other customers?
- How does this connect to PCA, which we covered in the previous case study?

### <font color="blue"> Data Dictionary

- **Customer_ID**: Customer identifier (300 customers)
- **Product_ID / Product_Name**: The product (41 products, same catalog as the Hierarchical Clustering case study)
- **Rating**: 1-5, how much that customer likes a product they've actually held/tried. Only observed for products a customer has actually used, most customer-product pairs are missing.

## <font color="brown"> Importing Necessary Libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import mean_squared_error

%matplotlib inline

## <font color="brown"> Part 1: SVD Is What Powers PCA

Before using SVD for something new, let's make the connection to the previous case study explicit. **Singular Value Decomposition** factors *any* matrix $X$ into three pieces:

$$X = U \Sigma V^T$$

- $U$: how each *row* (in our credit risk data, each customer) relates to the new axes
- $\Sigma$: a diagonal matrix of *singular values*, ranked largest to smallest, how much variance each new axis captures
- $V^T$: how each *column* (each original feature) contributes to the new axes

This is exactly what PCA needs. When $X$ is a *centered* (mean-subtracted), scaled data matrix, the columns of $V$ (equivalently, the rows of $V^T$) **are** the principal components, and the singular values in $\Sigma$ are directly related to the explained variance of each one. PCA doesn't use a separate algorithm, it's SVD applied to centered data. Let's verify this directly on the credit risk data from the PCA case study.

In [ ]:
credit_df = pd.read_csv(r"../PCA/credit_risk_profile.csv")
num_col = [c for c in credit_df.columns if c != 'Customer_ID']

from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(credit_df[num_col])

# Method 1: PCA
pca = PCA(n_components=3)
pca.fit(X_scaled)

# Method 2: raw SVD on the same (already centered by StandardScaler) matrix
U, S, Vt = np.linalg.svd(X_scaled, full_matrices=False)

# Compare PCA's components to SVD's V^T (allowing for sign flips, both are valid directions)
print("PCA component 1, first 5 values:", pca.components_[0][:5].round(4))
print("SVD Vt row 1, first 5 values:    ", Vt[0][:5].round(4))
print("SVD Vt row 1, sign-flipped:       ", (-Vt[0][:5]).round(4))

**The values match exactly (up to an arbitrary sign flip, both point along the same axis, just in opposite directions, which is mathematically equivalent and expected).** This confirms PCA really is just SVD on centered data, with the components relabeled as "principal components" and the singular values converted into explained variance.

## <font color="brown"> Part 2: SVD for Recommendations

PCA's use of SVD compresses *features*. SVD has an equally important second life: compressing **sparse matrices with missing values**, which is exactly what a customer-product ratings table looks like. This is the same core math, applied to a very different kind of problem.

## <font color="brown"> Reading the Ratings Data

In [ ]:
ratings_long = pd.read_csv("customer_product_ratings.csv")
ratings_long.shape

In [ ]:
ratings_long.head()

In [ ]:
products = pd.read_csv(r"../Hierarchical Clustering/banking_products.csv")
n_customers = ratings_long['Customer_ID'].nunique()
n_products = len(products)
possible_pairs = n_customers * n_products
print(f"{n_customers} customers x {n_products} products = {possible_pairs} possible pairs")
print(f"Actual ratings observed: {len(ratings_long)} ({len(ratings_long) / possible_pairs:.1%})")

**Only about 36% of all possible customer-product pairs have a rating.** This is completely typical of real recommendation problems, most customers interact with only a small fraction of any catalog. Our job is to fill in the other 64%, well enough to know which unrated products a customer would actually like.

### <font color="blue"> Building the Customer-Product Matrix

In [ ]:
customer_ids = sorted(ratings_long['Customer_ID'].unique())
product_ids = products['Product_ID'].tolist()

rating_matrix = ratings_long.pivot(
    index='Customer_ID', columns='Product_ID', values='Rating'
).reindex(index=customer_ids, columns=product_ids)

rating_matrix.iloc[:8, :8]

The blank (`NaN`) cells are exactly the products each customer hasn't rated, what we're trying to predict.

### <font color="blue"> Splitting Ratings for a Fair Evaluation

To honestly check how good our predictions are, we need to hide some *known* ratings from the model and see if it can guess them, using only the ratings it was allowed to see. This mirrors the same train/validation discipline we've used throughout, we never let the model peek at data it's being evaluated on.

In [ ]:
shuffled = ratings_long.sample(frac=1, random_state=1).reset_index(drop=True)
split_point = int(0.8 * len(shuffled))
train_long, val_long = shuffled.iloc[:split_point], shuffled.iloc[split_point:]

print("Train ratings:", len(train_long), " Validation ratings:", len(val_long))

In [ ]:
train_matrix = train_long.pivot(
    index='Customer_ID', columns='Product_ID', values='Rating'
).reindex(index=customer_ids, columns=product_ids)

# filling the remaining unknowns with each product's average training rating (a neutral starting point)
product_means = train_matrix.mean(axis=0)
global_mean = train_long['Rating'].mean()
filled_matrix = train_matrix.fillna(product_means).fillna(global_mean)

### <font color="blue"> Baseline: Just Guess the Product Average

In [ ]:
baseline_preds = val_long['Product_ID'].map(product_means).fillna(global_mean)
rmse_baseline = np.sqrt(mean_squared_error(val_long['Rating'], baseline_preds))
print(f"Naive baseline (product average) validation RMSE: {rmse_baseline:.4f}")

Before trying SVD, we need something to beat. The simplest possible recommender just predicts "whatever this product's average rating is," ignoring the customer entirely. Any real personalization should do better than this.

### <font color="blue"> Choosing the Rank with SVD

`TruncatedSVD` factors the matrix and immediately truncates it to keep only the top `k` singular values/vectors, this is the "low-rank approximation" idea: reconstruct the full matrix using only the `k` strongest underlying patterns, smoothing over noise and filling in the gaps.

In [ ]:
validation_rmse = {}

for k in [2, 3, 5, 8, 12, 18]:
    svd = TruncatedSVD(n_components=k, random_state=1)
    reduced = svd.fit_transform(filled_matrix)
    reconstructed = pd.DataFrame(
        reduced @ svd.components_, index=filled_matrix.index, columns=filled_matrix.columns
    )
    preds = [reconstructed.loc[r.Customer_ID, r.Product_ID] for r in val_long.itertuples()]
    rmse = np.sqrt(mean_squared_error(val_long['Rating'], preds))
    validation_rmse[k] = rmse
    print(f"k={k}: validation RMSE={rmse:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(list(validation_rmse.keys()), list(validation_rmse.values()), 'bx-', label='SVD')
plt.axhline(rmse_baseline, color='red', linestyle='--', label='Naive baseline')
plt.xlabel('Number of components (k)')
plt.ylabel('Validation RMSE')
plt.title('Choosing k by Validation RMSE')
plt.legend()
plt.show()

**RMSE improves sharply up to k=5, then gets *worse* as k keeps increasing.** This is overfitting, exactly the same lesson from PCA's scree plot, but showing up as a validation curve here instead: with too few components, we can't capture real customer-taste patterns; with too many, the model starts fitting noise introduced by our crude mean-filling of the unobserved cells, and validation performance actually degrades. **k=5 gives the best validation RMSE.**

### <font color="blue"> Final Model

In [ ]:
best_k = min(validation_rmse, key=validation_rmse.get)
print(f"Best k: {best_k}, validation RMSE: {validation_rmse[best_k]:.4f}")

svd_final = TruncatedSVD(n_components=best_k, random_state=1)
reduced_final = svd_final.fit_transform(filled_matrix)
predicted_ratings = pd.DataFrame(
    reduced_final @ svd_final.components_, index=filled_matrix.index, columns=filled_matrix.columns
)

improvement = (rmse_baseline - validation_rmse[best_k]) / rmse_baseline
print(f"Improvement over naive baseline: {improvement:.1%}")

## <font color="brown"> Making Recommendations

Let's see this in action for a specific customer: which products would we recommend that they haven't tried yet?

In [ ]:
example_customer = customer_ids[0]
already_rated = rating_matrix.loc[example_customer].dropna().index
not_yet_tried = rating_matrix.loc[example_customer][rating_matrix.loc[example_customer].isna()].index

recommendations = predicted_ratings.loc[example_customer, not_yet_tried].sort_values(ascending=False).head(5)
rec_df = products.set_index('Product_ID').loc[recommendations.index, ['Product_Name', 'Category']]
rec_df['Predicted_Rating'] = recommendations.values.round(2)
rec_df

These are the 5 highest predicted-rating products this customer hasn't tried, based purely on patterns learned from ratings across all 300 customers, not on any rule we wrote by hand about what "should" go together.

## <font color="brown"> Business Insights and Recommendations

- **SVD-based recommendations meaningfully beat the naive "most popular product" approach** (about a 13% reduction in prediction error), personalization based on each customer's own rating pattern genuinely adds value here, it isn't just noise.

- **Rank (k) needs to be tuned, not maximized.** More components isn't better, past k=5, performance gets worse, not just slower to compute. This is worth remembering any time SVD or PCA is used: always validate the rank choice, don't just pick a large number to "be safe."

- **This can directly power a cross-sell engine**: for any customer, sort their predicted ratings on products they haven't tried, and surface the top few in-branch, in-app, or by their relationship manager.

- **Sparsity is the real challenge, not the algorithm.** With only ~36% of pairs observed, collecting more ratings (even lightweight signals like "viewed product page" or "clicked learn more") would likely improve predictions further, there's a real ceiling on what any method can infer from this little data per customer.